# Oceanstream Geotrack Processing - Library API Demo

This notebook demonstrates using **oceanstream** as a library to process oceanographic data.

The `oceanstream.geotrack.process()` function accepts the same parameters as the CLI command:
```bash
oceanstream process geotrack --input-dir ./raw_data --output-dir ./out --yes -v
```

This example shows how to call the same functionality programmatically.

## 1. Setup and Imports

## CLI Equivalent

The library API mirrors the CLI interface:

**CLI:**
```bash
oceanstream process geotrack \
  --input-dir ./raw_data \
  --output-dir ./out/geoparquet \
  --yes \
  -v
```

**Library API:**
```python
from oceanstream.geotrack import process
from oceanstream.providers import get_provider

provider = get_provider("saildrone")
process(
    provider=provider,
    input_dir="./raw_data",
    output_dir="./out/geoparquet",
    yes=True,
    verbose=True
)
```

In [ ]:
import sys
from pathlib import Path

# Add the project root to the path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import oceanstream modules
from oceanstream.geotrack import process
from oceanstream.providers import get_provider

print("✅ Imports successful!")
print(f"📁 Project root: {project_root}")

## 2. Setup Input and Output Directories

In [ ]:
import tempfile

# Input: Use test data from the project
input_dir = project_root / "oceanstream" / "tests" / "data" / "raw_data"

# Output: Create a temporary directory
output_dir = Path(tempfile.mkdtemp())

print(f"📂 Input directory:  {input_dir}")
print(f"📂 Output directory: {output_dir}")

# Verify input data exists
if input_dir.exists():
    csv_files = list(input_dir.glob("*.csv"))
    print(f"\n✅ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"   • {f.name}")
else:
    print("\n❌ Input directory not found!")

## 3. Initialize Provider

In [ ]:
# Get the Saildrone provider
provider = get_provider("saildrone")

print(f"✅ Provider: {provider.name}")
print(f"📋 Supported modules: {provider.supported_modules}")

## 4. Process Geotrack Data

Call `process()` with the same parameters as the CLI.

In [ ]:
print("🚀 Processing geotrack data...\n")

# Call the process function - same parameters as CLI:
# oceanstream process geotrack --input-dir <input_dir> --output-dir <output_dir> --yes -v
process(
    provider=provider,
    input_dir=input_dir,
    output_dir=output_dir,
    verbose=True,      # -v flag
    yes=True,          # --yes flag (skip confirmation)
)

print("\n✅ Processing complete!")

## 5. Generate PMTiles (Optional)

You can also generate PMTiles vector tiles from the GeoParquet output for web mapping applications. This requires `ogr2ogr` (GDAL) and the `pmtiles` CLI to be installed.

**CLI:**
```bash
oceanstream process geotrack \
  --input-dir ./raw_data \
  --output-dir ./out/geoparquet \
  --generate-pmtiles \
  --pmtiles-minzoom 0 \
  --pmtiles-maxzoom 10 \
  --pmtiles-layer oceanstream_track \
  --yes -v
```

**Library API:**
```python
process(
    provider=provider,
    input_dir=input_dir,
    output_dir=output_dir,
    generate_pmtiles=True,
    pmtiles_minzoom=0,
    pmtiles_maxzoom=10,
    pmtiles_layer="oceanstream_track",
    yes=True,
    verbose=True
)
```

The PMTiles file will be created at `{output_dir}/track.pmtiles` and can be served directly for MapLibre/Mapbox GL JS web maps.

## 5. Inspect the Output

Examine the generated GeoParquet files.

In [ ]:
import pyarrow.parquet as pq

# List all parquet files
parquet_files = list(output_dir.rglob("*.parquet"))

print(f"📊 Generated {len(parquet_files)} parquet file(s)\n")

if parquet_files:
    # Show partition structure
    print("📁 Partition structure:")
    for lat_dir in sorted(output_dir.iterdir()):
        if lat_dir.is_dir() and lat_dir.name.startswith("lat_bin="):
            print(f"   {lat_dir.name}/")
            for lon_dir in sorted(lat_dir.iterdir()):
                if lon_dir.is_dir() and lon_dir.name.startswith("lon_bin="):
                    files = list(lon_dir.glob("*.parquet"))
                    print(f"      {lon_dir.name}/ ({len(files)} file(s))")
    
    # Read and display sample data
    sample_file = sorted(parquet_files)[0]
    print(f"\n📄 Sample file: {sample_file.relative_to(output_dir)}")
    
    table = pq.read_table(sample_file)
    df = table.to_pandas()
    
    print(f"\n   Shape: {df.shape}")
    print(f"   Columns ({len(df.columns)}): {', '.join(df.columns[:10])}{'...' if len(df.columns) > 10 else ''}")
    print(f"\n   First 3 rows:")
    print(df[["latitude", "longitude", "time"]].head(3))
else:
    print("❌ No parquet files found!")

## 6. Verify Metadata

Check the GeoParquet metadata and column aliases.

In [ ]:
import json

if parquet_files:
    # Read metadata from the sample file
    parquet_file = pq.ParquetFile(sample_file)
    metadata = parquet_file.schema_arrow.metadata
    
    print("🔍 Parquet Metadata:\n")
    
    # Show GeoParquet metadata
    if b'geo' in metadata:
        geo_meta = json.loads(metadata[b'geo'])
        print(f"📍 GeoParquet version: {geo_meta.get('version')}")
        print(f"📍 Primary geometry column: {geo_meta.get('primary_column')}")
        print(f"📍 CRS: {geo_meta['columns']['geometry']['crs']['id']['code']}")
    
    # Show column aliases (semantic mappings)
    if b'oceanstream:aliases' in metadata:
        aliases = json.loads(metadata[b'oceanstream:aliases'])
        print(f"\n📋 Column aliases (first 5):")
        for orig, alias in list(aliases.items())[:5]:
            print(f"   {orig:20} → {alias}")
        if len(aliases) > 5:
            print(f"   ... and {len(aliases) - 5} more")
    
    # Show provider metadata
    if b'oceanstream:provider' in metadata:
        provider_meta = json.loads(metadata[b'oceanstream:provider'])
        print(f"\n🏷️  Provider: {provider_meta.get('name')}")
else:
    print("❌ No files to inspect")

## 7. Cleanup

In [ ]:
import shutil

# Clean up temporary output directory
if output_dir.exists():
    shutil.rmtree(output_dir)
    print(f"🗑️  Cleaned up: {output_dir}")

print("\n✅ Demo complete!")